# Diabetes Risk Prediction — Preprocessing Pipeline

Converts raw hospital data into ML-ready arrays.

**Key design decision:** `diag_1`, `diag_2`, `diag_3` each have 700+ unique ICD-9 codes.
One-hot encoding them directly produces 2300+ features, which causes memory errors.

**Solution:** Map ICD-9 codes to ~20 clinical disease categories
(Circulatory, Respiratory, Diabetes, etc.) — reducing features to a manageable size
while preserving clinical meaning.

In [1]:
!pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 1 — Load Dataset

In [3]:
df = pd.read_csv('../data/raw/diabetic_data.csv')

print('Dataset shape:', df.shape)
print('Target class distribution:')
print(df['readmitted'].value_counts())

Dataset shape: (101766, 50)
Target class distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


## Step 2 — Clean Raw Data

In [4]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Drop high-missing columns
drop_cols = [
    'weight', 'payer_code', 'medical_specialty',
    'max_glu_serum', 'A1Cresult',
    'encounter_id', 'patient_nbr'
]
df.drop(columns=drop_cols, inplace=True)

# Fill remaining missing values with mode
for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print('Shape after cleaning:', df.shape)
print('Remaining missing values:', df.isnull().sum().sum())

Shape after cleaning: (101766, 43)
Remaining missing values: 0


## Step 3 — Map ICD-9 Diagnosis Codes to Disease Categories

Instead of one-hot encoding 700+ diagnosis codes (which creates thousands of columns),
we map them to ~20 standard clinical disease groups.

This follows the approach used in published diabetes readmission research
(Strack et al., 2014 — the original paper for this dataset).

In [5]:
def map_icd9_to_category(code):
    """Map an ICD-9 diagnosis code to a clinical disease category."""
    if pd.isna(code):
        return 'Other'
    code = str(code).strip()
    # Remove leading 'V' and 'E' codes
    if code.startswith('V'):
        return 'Supplementary'
    if code.startswith('E'):
        return 'Injury'
    try:
        num = float(code)
    except ValueError:
        return 'Other'

    if 390 <= num <= 459 or num == 785:
        return 'Circulatory'
    elif 460 <= num <= 519 or num == 786:
        return 'Respiratory'
    elif 520 <= num <= 579 or num == 787:
        return 'Digestive'
    elif num == 250:
        return 'Diabetes'
    elif 800 <= num <= 999:
        return 'Injury'
    elif 710 <= num <= 739:
        return 'Musculoskeletal'
    elif 580 <= num <= 629 or num == 788:
        return 'Genitourinary'
    elif 140 <= num <= 239:
        return 'Neoplasms'
    elif 240 <= num <= 279:
        return 'Endocrine'
    elif 680 <= num <= 709 or num == 782:
        return 'Skin'
    elif 290 <= num <= 319:
        return 'Mental'
    elif 1 <= num <= 139:
        return 'Infectious'
    elif 280 <= num <= 289:
        return 'Blood'
    elif 320 <= num <= 389:
        return 'Nervous'
    elif 630 <= num <= 679:
        return 'Pregnancy'
    elif 740 <= num <= 759:
        return 'Congenital'
    elif 760 <= num <= 779:
        return 'Perinatal'
    elif 780 <= num <= 799:
        return 'Symptoms'
    else:
        return 'Other'

# Apply to all three diagnosis columns
for col in ['diag_1', 'diag_2', 'diag_3']:
    new_col = col + '_cat'
    df[new_col] = df[col].apply(map_icd9_to_category)
    print(f'{col} → {new_col}  (unique categories: {df[new_col].nunique()})')

# Drop original diagnosis columns
df.drop(columns=['diag_1', 'diag_2', 'diag_3'], inplace=True)

print('\nShape after diagnosis mapping:', df.shape)

diag_1 → diag_1_cat  (unique categories: 18)
diag_2 → diag_2_cat  (unique categories: 18)
diag_3 → diag_3_cat  (unique categories: 18)

Shape after diagnosis mapping: (101766, 43)


## Step 4 — Drop Near-Constant Medication Columns

Medication columns that are >99% the same value have virtually no signal.
Examples: `examide`, `citoglipton`, `acetohexamide` (all 'No' or one value).

In [6]:
obj_cols = [c for c in df.select_dtypes(include='object').columns if c != 'readmitted']

near_constant = []
for col in obj_cols:
    top_freq = df[col].value_counts(normalize=True).iloc[0]
    if top_freq >= 0.99:
        near_constant.append(col)

print(f'Near-constant columns dropped ({len(near_constant)}): {near_constant}')

df.drop(columns=near_constant, inplace=True)

print('\nShape after dropping near-constant columns:', df.shape)
print('Remaining object columns:', df.select_dtypes(include='object').columns.tolist())

Near-constant columns dropped (15): ['nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

Shape after dropping near-constant columns: (101766, 28)
Remaining object columns: ['race', 'gender', 'age', 'metformin', 'repaglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'change', 'diabetesMed', 'readmitted', 'diag_1_cat', 'diag_2_cat', 'diag_3_cat']


## Step 5 — Encode Target Variable

| Original | Risk Level | Code |
|---|---|---|
| NO | Low Risk | 0 |
| >30 | Medium Risk | 1 |
| <30 | High Risk | 2 |

In [7]:
risk_mapping = {'NO': 0, '>30': 1, '<30': 2}
df['readmitted'] = df['readmitted'].map(risk_mapping)

print('Class distribution after encoding:')
print(df['readmitted'].value_counts())
print('0=Low Risk | 1=Medium Risk | 2=High Risk')

Class distribution after encoding:
readmitted
0    54864
1    35545
2    11357
Name: count, dtype: int64
0=Low Risk | 1=Medium Risk | 2=High Risk


## Step 6 — Separate X and y

In [8]:
X = df.drop('readmitted', axis=1)
y = df['readmitted']

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (101766, 27)
y shape: (101766,)


## Step 7 — One-Hot Encode Categorical Features

With diagnosis codes mapped to categories, the total feature count
is now small enough to handle efficiently.

In [9]:
X = pd.get_dummies(X, drop_first=True)

print('Shape after one-hot encoding:', X.shape)
print('Total features:', X.shape[1])

Shape after one-hot encoding: (101766, 103)
Total features: 103


## Step 8 — Train / Test Split (80/20, stratified)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training set:', X_train.shape)
print('Testing  set:', X_test.shape)
print('\nClass distribution in y_train:')
print(y_train.value_counts())

Training set: (81412, 103)
Testing  set: (20354, 103)

Class distribution in y_train:
readmitted
0    43891
1    28436
2     9085
Name: count, dtype: int64


## Step 9 — Scale Numerical Features

Fit scaler on training data ONLY, then transform both.

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train.astype(np.float32))
X_test_scaled  = scaler.transform(X_test.astype(np.float32))

print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled  shape:', X_test_scaled.shape)
print(f'Memory (train dense): {X_train_scaled.nbytes / 1e6:.1f} MB')

X_train_scaled shape: (81412, 103)
X_test_scaled  shape: (20354, 103)
Memory (train dense): 33.5 MB


## Step 10 — Apply SMOTE

Balance training classes synthetically.
Applied ONLY on training data — never test data.

In [12]:
print('Class distribution BEFORE SMOTE:')
print(y_train.value_counts())
print()

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled,
    y_train
)

print('Class distribution AFTER SMOTE:')
print(pd.Series(y_train_resampled).value_counts())
print()
print('X_train_resampled shape:', X_train_resampled.shape)

Class distribution BEFORE SMOTE:
readmitted
0    43891
1    28436
2     9085
Name: count, dtype: int64

Class distribution AFTER SMOTE:
readmitted
0    43891
2    43891
1    43891
Name: count, dtype: int64

X_train_resampled shape: (131673, 103)


## Step 11 — Save All Artefacts

In [13]:
os.makedirs('../models', exist_ok=True)

# Scaler and feature names (for API inference)
joblib.dump(scaler,          '../models/scaler.pkl')
joblib.dump(list(X.columns), '../models/feature_columns.pkl')

# Processed training data (SMOTE-balanced)
joblib.dump(X_train_resampled, '../models/X_train_resampled.pkl')
joblib.dump(y_train_resampled, '../models/y_train_resampled.pkl')

# Test data
joblib.dump(X_test_scaled, '../models/X_test_scaled.pkl')
joblib.dump(y_test,        '../models/y_test.pkl')

print('Saved artefacts:')
for f in ['scaler.pkl', 'feature_columns.pkl',
          'X_train_resampled.pkl', 'y_train_resampled.pkl',
          'X_test_scaled.pkl', 'y_test.pkl']:
    size = os.path.getsize(f'../models/{f}') / 1e6
    print(f'  ../models/{f}  ({size:.1f} MB)')

print()
print('Preprocessing complete! Ready for model training.')

Saved artefacts:
  ../models/scaler.pkl  (0.0 MB)
  ../models/feature_columns.pkl  (0.0 MB)
  ../models/X_train_resampled.pkl  (54.2 MB)
  ../models/y_train_resampled.pkl  (2.1 MB)
  ../models/X_test_scaled.pkl  (8.4 MB)
  ../models/y_test.pkl  (0.7 MB)

Preprocessing complete! Ready for model training.
